# mT5 LoRA fine-tuning — MWE paraphrasing (PARSEME 2.0 Subtask 2)

Multilingual seq2seq SFT with PEFT/LoRA on `google/mt5-large`.

- **Input format:** `paraphrase <LANG> [idiom: {idiom}]: {sentence}`
- **Target:** `{paraphrase}`
- **Data sources:** `gemini_outputs.json` + `synthetic_data.json` (merged, deduplicated)
- **Save location:** `MyDrive/mt5_mwe/checkpoints/`
- **Tier assumed:** Colab Pro (L4 24 GB or A100 40 GB)

Runtime → *Change runtime type* → GPU (L4 or A100) before running.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.41" "peft>=0.11" "accelerate>=0.30" \
                "datasets>=2.18" "sentencepiece" "tqdm" "bert-score"
# Colab ships an old torchao (0.10.0) that current PEFT rejects on import.
# We don't use torchao — uninstall so PEFT's dispatcher silently skips that path.
!pip -q uninstall -y torchao

## 2. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/mt5_mwe'
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
CKPT_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapter_final')
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
print('Data dir   :', DATA_DIR)
print('Checkpoints:', CKPT_DIR)

## 3. Load training data

Upload **one or both** files to `MyDrive/mt5_mwe/data/` on Drive, then run this cell.  
Both share the same `{idiom, example, paraphrase}` schema.  
Duplicates (same language + sentence) are removed, with `gemini_outputs.json` entries taking priority.

| File | Languages | Pairs |
|---|---|---|
| `gemini_outputs.json` | 12 (no PT, SV) | ~491 |
| `synthetic_data.json` | 14 | ~980 |

In [ ]:
import json
from collections import Counter

LANG_MAP = {
    'French': 'FR', 'Georgian': 'KA', 'Greek': 'EL', 'Japanese': 'JA',
    'Hebrew': 'HE', 'Latvian': 'LV', 'Persian': 'FA', 'Polish': 'PL',
    'Romanian': 'RO', 'Serbian': 'SR', 'Slovene': 'SL', 'Ukrainian': 'UK',
    'Portuguese': 'PT', 'Brazilian Portuguese': 'PT', 'Swedish': 'SV',
}

def _load_source(filename):
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        print(f'  (skipping {filename} — not found)')
        return {}
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def _extract_pairs(data, source_label):
    out = []
    for lang_name, records in data.items():
        code = LANG_MAP.get(lang_name)
        if code is None:
            print(f'  ! [{source_label}] unknown language "{lang_name}" — skipped')
            continue
        for r in records:
            s     = (r.get('example')    or '').strip()
            p     = (r.get('paraphrase') or '').strip()
            idiom = (r.get('idiom')      or '').strip()
            if s and p and s != p:
                out.append({'language': code, 'sentence': s, 'paraphrase': p, 'idiom': idiom})
    return out

gemini_pairs    = _extract_pairs(_load_source('gemini_outputs.json'),  'gemini')
synthetic_pairs = _extract_pairs(_load_source('synthetic_data.json'),  'synthetic')

assert gemini_pairs or synthetic_pairs, (
    'Neither gemini_outputs.json nor synthetic_data.json found in MyDrive/mt5_mwe/data/. '
    'Upload at least one file to Google Drive first.'
)

# Merge: gemini first (higher quality), synthetic adds new pairs only
seen  = {(r['language'], r['sentence'].lower()) for r in gemini_pairs}
pairs = list(gemini_pairs)
added = 0
for r in synthetic_pairs:
    key = (r['language'], r['sentence'].lower())
    if key not in seen:
        seen.add(key)
        pairs.append(r)
        added += 1

print(f'gemini_outputs.json : {len(gemini_pairs)} pairs')
print(f'synthetic_data.json : {len(synthetic_pairs)} pairs  ({added} new after dedup)')
print(f'Total merged        : {len(pairs)} pairs')
print('By language:', dict(Counter(r['language'] for r in pairs)))
assert len(pairs) > 0, 'No valid pairs found.'

## 4. Hyperparameters

Tune these in one place. Defaults target Colab Pro on L4 24 GB.

In [ ]:
MODEL_NAME       = 'google/mt5-large'   # try 'google/mt5-base' if OOM, 'google/mt5-xl' on A100 40GB
MAX_INPUT_LEN    = 192
MAX_TARGET_LEN   = 192

# LoRA
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGETS     = ['q', 'v']           # q/v projections of attention

# Optimization
EPOCHS           = 5
PER_DEVICE_BS    = 8
GRAD_ACCUM       = 4                    # effective batch 32
LR               = 3e-4                 # standard for LoRA; ~10x higher than full-FT
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
LABEL_SMOOTHING  = 0.1

# Eval/save
VAL_FRACTION     = 0.10
SEED             = 42
LOG_STEPS        = 10
SAVE_TOTAL_LIMIT = 2

## 5. Train/val split

In [ ]:
import random
from collections import defaultdict
from datasets import Dataset

random.seed(SEED)

def to_io(r):
    return {
        'input_text' : f"paraphrase <{r['language']}> [idiom: {r['idiom']}]: {r['sentence']}",
        'target_text': r['paraphrase'],
    }

# Stratify by language so every language gets at least 1 val example
by_lang = defaultdict(list)
for r in pairs:
    by_lang[r['language']].append(r)

train_set, val_set = [], []
print(f'Per-language split (val_fraction={VAL_FRACTION}, min 1 val):')
for lang in sorted(by_lang):
    items = by_lang[lang]
    random.shuffle(items)
    n_val_lang = max(1, round(len(items) * VAL_FRACTION))
    val_set.extend(  to_io(r) for r in items[:n_val_lang])
    train_set.extend(to_io(r) for r in items[n_val_lang:])
    print(f'  {lang}: train={len(items) - n_val_lang:3d}, val={n_val_lang}')

# Shuffle so languages are interleaved during training/eval
random.shuffle(train_set)
random.shuffle(val_set)
print(f'\nTotals — train: {len(train_set)} | val: {len(val_set)}')

train_ds = Dataset.from_list(train_set)
val_ds   = Dataset.from_list(val_set)

## 6. Tokenizer + tokenization

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print('Vocab size:', tokenizer.vocab_size)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(  tokenize_batch, batched=True, remove_columns=val_ds.column_names)
print(train_tok)

## 7. Load mT5 + apply LoRA

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype    = torch.bfloat16 if use_bf16 else torch.float16
print('CUDA  :', torch.cuda.is_available(), '| bf16:', use_bf16)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))

base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGETS,
    bias           = 'none',
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# Required for LoRA + gradient_checkpointing: lets gradients flow from the
# (frozen) input embeddings into the adapter weights during the recomputed
# backward pass. Newer PEFT versions handle this in get_peft_model, but
# being explicit avoids silent gradient loss on older versions.
model.enable_input_require_grads()

## 8. Trainer setup

`Seq2SeqTrainer` already shows a tqdm progress bar per epoch.

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest')

args = Seq2SeqTrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BS,
    per_device_eval_batch_size  = PER_DEVICE_BS,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    label_smoothing_factor      = LABEL_SMOOTHING,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    bf16                        = use_bf16,
    fp16                        = (not use_bf16) and torch.cuda.is_available(),
    gradient_checkpointing      = True,
    logging_steps               = LOG_STEPS,
    predict_with_generate       = False,    # speed up eval; we'll generate manually after
    report_to                   = 'none',
    seed                        = SEED,
    disable_tqdm                = False,
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = train_tok,
    eval_dataset  = val_tok,
    processing_class = tokenizer,
    data_collator = collator,
)

## 9. Train

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 10. Save the LoRA adapter, tokenizer, and run config

Adapter weights only — you load them on top of stock `google/mt5-large` later via `PeftModel.from_pretrained`.

In [ ]:
import json, datetime

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run_meta = {
    'model_name'      : MODEL_NAME,
    'lora'            : {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
                          'targets': LORA_TARGETS},
    'epochs'          : EPOCHS,
    'effective_bs'    : PER_DEVICE_BS * GRAD_ACCUM,
    'lr'              : LR,
    'max_input_len'   : MAX_INPUT_LEN,
    'max_target_len'  : MAX_TARGET_LEN,
    'n_train'         : len(train_tok),
    'n_val'           : len(val_tok),
    'languages'       : sorted(set(r['language'] for r in pairs)),
    'final_metrics'   : train_result.metrics,
    'saved_at'        : datetime.datetime.utcnow().isoformat() + 'Z',
}
with open(os.path.join(ADAPTER_DIR, 'run_meta.json'), 'w') as f:
    json.dump(run_meta, f, indent=2, default=str)

print('Saved adapter \u2192', ADAPTER_DIR)
print('Files       :', os.listdir(ADAPTER_DIR))

## 11. Inference smoke test

Generate paraphrases for the first few validation examples to sanity-check the trained adapter.

In [ ]:
from tqdm.auto import tqdm

model.eval()
device = next(model.parameters()).device
n_show = min(8, len(val_set))

print(f'\nGenerating {n_show} samples on {device}\n' + '\u2500' * 60)
for ex in tqdm(val_set[:n_show], desc='generating'):
    enc = tokenizer(ex['input_text'], return_tensors='pt',
                    max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens = MAX_TARGET_LEN,
            num_beams      = 4,
            early_stopping = True,
            no_repeat_ngram_size = 3,
        )
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"INPUT : {ex['input_text']}")
    print(f"GOLD  : {ex['target_text']}")
    print(f"PRED  : {pred}")
    print('\u2500' * 60)

## 12. Generate Codabench submission files

Reads `MyDrive/mt5_mwe/data/<LANG>/test.blind.json` for each of the 14 languages, runs batched generation, writes `MyDrive/mt5_mwe/system_predictions/<LANG>/test.system.json`, and zips them into `MyDrive/mt5_mwe/submission.zip` with the nested `<LANG>/test.system.json` layout Codabench expects.

Prompt format follows what the model was trained on:
- **FR**: `paraphrase <FR> [idiom: {mwe}]: {sentence}` — `mwe` is extracted from the `[[…]]` brackets in `raw_text`, then brackets are stripped from the sentence.
- **Other 13 languages**: `paraphrase <LANG>: {sentence}` — no idiom hint available in the test files. Quality on these languages will be lower than FR.

In [ ]:
import json, re, zipfile
from pathlib import Path
from tqdm.auto import tqdm

TEST_DIR   = Path(DRIVE_ROOT) / 'data'
SYSTEM_DIR = Path(DRIVE_ROOT) / 'system_predictions'
ZIP_PATH   = Path(DRIVE_ROOT) / 'submission.zip'
SYSTEM_DIR.mkdir(parents=True, exist_ok=True)

LANG_CODES   = sorted(set(LANG_MAP.values()))    # 14 codes
BRACKET_RE   = re.compile(r'\[\[(.*?)\]\]')
GEN_BATCH_SZ = 8

def format_input(lang_code: str, raw_text: str) -> str:
    if lang_code == 'FR':
        m = BRACKET_RE.search(raw_text)
        idiom = m.group(1).strip() if m else ''
        clean = raw_text.replace('[[', '').replace(']]', '').strip()
        if idiom:
            return f'paraphrase <FR> [idiom: {idiom}]: {clean}'
        return f'paraphrase <FR>: {clean}'
    return f'paraphrase <{lang_code}>: {raw_text.strip()}'

def batch_generate(texts, batch_size=GEN_BATCH_SZ):
    model.eval()
    device = next(model.parameters()).device
    outs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='generate', leave=False):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_INPUT_LEN).to(device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens       = MAX_TARGET_LEN,
                num_beams            = 4,
                early_stopping       = True,
                no_repeat_ngram_size = 3,
            )
        outs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return outs

written = []
for code in LANG_CODES:
    in_path = TEST_DIR / code / 'test.blind.json'
    if not in_path.exists():
        print(f'  ! missing {in_path} — skipping {code}')
        continue
    with open(in_path, encoding='utf-8') as f:
        records = json.load(f)
    print(f'\n[{code}] {len(records)} records')
    preds = batch_generate([format_input(code, r['raw_text']) for r in records])
    out_records = [{'source_sent_id': r['source_sent_id'], 'prediction': p}
                   for r, p in zip(records, preds)]
    out_path = SYSTEM_DIR / code / 'test.system.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(out_records, f, ensure_ascii=False, indent=2)
    print(f'  → wrote {out_path}  ({len(out_records)} predictions)')
    written.append(code)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for code in written:
        zf.write(SYSTEM_DIR / code / 'test.system.json',
                 arcname=f'{code}/test.system.json')
print(f'\nZipped {len(written)} languages → {ZIP_PATH}')

## 13. (Optional) Reload later from Drive

After a runtime restart, you don't need to retrain — just reload base + adapter:

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-large', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, '/content/drive/MyDrive/mt5_mwe/adapter_final')
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/mt5_mwe/adapter_final')
```